In [3]:
# 02_multi_compartment.ipynb – Two‑Compartment Lung Model 

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from IPython.display import display, clear_output

# ============================================================
# Two‑compartment ODE model
# ============================================================

def two_comp_ode(t, y, R1, C1, R2, C2, P_insp, PEEP, insp_time, rate):
    V1, V2 = y
    period = 60 / rate
    t_mod = t % period
    P_aw = P_insp if t_mod < insp_time else 0.0
    Q1 = (P_aw - V1/C1) / R1
    Q2 = (P_aw - V2/C2) / R2
    if V1 <= 0 and Q1 < 0:
        Q1 = 0
    if V2 <= 0 and Q2 < 0:
        Q2 = 0
    return [Q1, Q2]

def simulate_two(R1, C1, R2, C2, P_insp, PEEP, rate, insp_frac=0.33, duration=10):
    insp_time = (60/rate) * insp_frac
    t_eval = np.linspace(0, duration, 2000)
    sol = solve_ivp(two_comp_ode, (0, duration), [0.0, 0.0], t_eval=t_eval,
                    args=(R1, C1, R2, C2, P_insp, PEEP, insp_time, rate),
                    method='RK45', rtol=1e-6)
    t = sol.t
    V1, V2 = sol.y
    Q1 = np.gradient(V1, t)
    Q2 = np.gradient(V2, t)
    Vtot = V1 + V2
    P_aw = PEEP + V1/C1 + R1 * Q1   # same pressure for both
    return t, V1, V2, Vtot, Q1, Q2, P_aw

def compute_metrics_two(t, V1, V2, P_aw, PEEP, rate):
    Vt1 = np.max(V1) - np.min(V1)
    Vt2 = np.max(V2) - np.min(V2)
    Vt_tot = Vt1 + Vt2
    minute_vent = Vt_tot * rate
    peak_P = np.max(P_aw)
    mean_P = np.mean(P_aw)
    C_dyn_tot = Vt_tot / (peak_P - PEEP) if (peak_P - PEEP) > 0 else 0
    # Asynchrony: difference in time to reach peak volume (seconds)
    t_peak1 = t[np.argmax(V1)] if len(V1) > 0 else 0
    t_peak2 = t[np.argmax(V2)] if len(V2) > 0 else 0
    asynchrony = abs(t_peak1 - t_peak2)
    return {
        'Vt1_mL': Vt1 * 1000,
        'Vt2_mL': Vt2 * 1000,
        'Vt_total_mL': Vt_tot * 1000,
        'Minute_vent_Lmin': minute_vent,
        'Peak_P_cmH2O': peak_P,
        'Mean_P_cmH2O': mean_P,
        'C_dyn_total_mL_cmH2O': C_dyn_tot * 1000,
        'Asynchrony_s': asynchrony
    }

def plot_results_two(t, V1, V2, Vtot, Q1, Q2, P_aw, R1, C1, R2, C2, P_insp, PEEP, rate, insp_frac):
    metrics = compute_metrics_two(t, V1, V2, P_aw, PEEP, rate)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
    ax1, ax2 = axes[0, 0], axes[0, 1]
    ax3, ax4 = axes[1, 0], axes[1, 1]
    
    # Airway pressure
    ax1.plot(t, P_aw, 'b-', lw=2)
    ax1.axhline(PEEP, color='gray', linestyle='--', label=f'PEEP = {PEEP} cmH2O')
    ax1.set_ylabel('Pressure (cmH2O)')
    ax1.set_title('Airway Pressure')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Compartment volumes
    ax2.plot(t, V1*1000, 'r-', lw=2, label='Compartment 1')
    ax2.plot(t, V2*1000, 'g-', lw=2, label='Compartment 2')
    ax2.plot(t, Vtot*1000, 'k--', lw=1.5, label='Total')
    ax2.set_ylabel('Volume (mL)')
    ax2.set_title('Lung Volumes')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Compartment flows
    ax3.plot(t, Q1, 'r-', lw=2, label='Flow 1')
    ax3.plot(t, Q2, 'g-', lw=2, label='Flow 2')
    ax3.axhline(0, color='k', lw=0.5)
    ax3.set_xlabel('Time (s)')
    ax3.set_ylabel('Flow (L/s)')
    ax3.set_title('Flow Waveforms')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Pressure‑Volume loops for both compartments
    ax4.plot(P_aw, V1*1000, 'r-', lw=1.5, label='Comp 1 PV loop')
    ax4.plot(P_aw, V2*1000, 'g-', lw=1.5, label='Comp 2 PV loop')
    ax4.set_xlabel('Pressure (cmH2O)')
    ax4.set_ylabel('Volume (mL)')
    ax4.set_title('Pressure‑Volume Loops')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    fig.suptitle(f'Two‑Compartment Model: R1={R1:.1f}, C1={C1:.3f}, R2={R2:.1f}, C2={C2:.3f}, Rate={rate} bpm',
                 fontsize=12)
    
    # Metrics text box
    textstr = (f"Tidal Volumes: C1 = {metrics['Vt1_mL']:.0f} mL, C2 = {metrics['Vt2_mL']:.0f} mL\n"
               f"Total Vt: {metrics['Vt_total_mL']:.0f} mL\n"
               f"Minute Ventilation: {metrics['Minute_vent_Lmin']:.2f} L/min\n"
               f"Peak Pressure: {metrics['Peak_P_cmH2O']:.1f} cmH2O\n"
               f"Dynamic Compliance (total): {metrics['C_dyn_total_mL_cmH2O']:.1f} mL/cmH2O\n"
               f"Asynchrony (time shift): {metrics['Asynchrony_s']:.3f} s")
    
    ax1.text(0.02, 0.98, textstr, transform=ax1.transAxes, fontsize=9,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.show()
    
    print("\n📊 Clinical Metrics (Two Compartments):")
    for k, v in metrics.items():
        print(f"  {k}: {v:.2f}")

# ============================================================
# Interactive widgets
# ============================================================

style = {'description_width': 'initial'}

# Compartment 1
R1_slider = widgets.FloatSlider(value=8, min=2, max=30, step=0.5, description='R1 (cmH2O/L/s):', style=style)
C1_slider = widgets.FloatSlider(value=0.06, min=0.02, max=0.15, step=0.002, description='C1 (L/cmH2O):', style=style)
# Compartment 2
R2_slider = widgets.FloatSlider(value=12, min=2, max=30, step=0.5, description='R2 (cmH2O/L/s):', style=style)
C2_slider = widgets.FloatSlider(value=0.04, min=0.02, max=0.15, step=0.002, description='C2 (L/cmH2O):', style=style)

P_insp_slider = widgets.FloatSlider(value=15, min=5, max=30, step=0.5, description='Insp Pressure (cmH2O):', style=style)
PEEP_slider = widgets.FloatSlider(value=5, min=0, max=15, step=1, description='PEEP (cmH2O):', style=style)
rate_slider = widgets.IntSlider(value=15, min=8, max=30, step=1, description='Rate (bpm):', style=style)
insp_frac_slider = widgets.FloatSlider(value=0.33, min=0.2, max=0.5, step=0.01, description='I:E ratio (insp fraction):', style=style)
duration_slider = widgets.FloatSlider(value=10, min=5, max=30, step=1, description='Duration (s):', style=style)

def update_two(R1, C1, R2, C2, P_insp, PEEP, rate, insp_frac, duration):
    clear_output(wait=True)
    t, V1, V2, Vtot, Q1, Q2, P_aw = simulate_two(R1, C1, R2, C2, P_insp, PEEP, rate, insp_frac, duration)
    plot_results_two(t, V1, V2, Vtot, Q1, Q2, P_aw, R1, C1, R2, C2, P_insp, PEEP, rate, insp_frac)

ui_two = widgets.VBox([
    widgets.HBox([R1_slider, C1_slider]),
    widgets.HBox([R2_slider, C2_slider]),
    widgets.HBox([P_insp_slider, PEEP_slider]),
    widgets.HBox([rate_slider, insp_frac_slider]),
    duration_slider
])

out_two = widgets.interactive_output(update_two, {
    'R1': R1_slider, 'C1': C1_slider, 'R2': R2_slider, 'C2': C2_slider,
    'P_insp': P_insp_slider, 'PEEP': PEEP_slider, 'rate': rate_slider,
    'insp_frac': insp_frac_slider, 'duration': duration_slider
})

display(ui_two, out_two)

Output()